# Spectral-Semantic Fusion for AI Image Detection

This notebook implements a two-stage feature extraction and classification pipeline as per the research architecture:
1.  **Low-level Features**: Spectral analysis via 2D-FFT and Radial Profiling.
2.  **High-level Features**: Semantic consistency via CLIP (Contrastive Language-Image Pretraining) embeddings.
3.  **Fusion & Classification**: Concatenation of features followed by SVM and XGBoost classifiers.

**Target Dataset**: GenImage style (Real vs. AI-generated).

In [ ]:
import sys
import os

# Verify the current active environment (expected: 'edp')
print(f"Current Python Prefix: {sys.prefix}")
!conda info --envs | grep "*"

# Verify core system info
print(f"Platform: {sys.platform}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier
from tqdm import tqdm

print("Libraries imported successfully.")
print(f"PyTorch Version: {torch.__version__}")
# Device setup
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
class SpectralFeatureExtractor:
    """Extracts 1D Radial Profile from 2D FFT Magnitude Spectrum."""
    def __init__(self, target_size=256):
        self.target_size = target_size
        self.num_bins = target_size // 2

    def _azimuthal_integration(self, magnitude_spectrum):
        """Computes the radial profile (azimuthal average) of a 2D magnitude spectrum."""
        y, x = np.indices(magnitude_spectrum.shape)
        center = np.array(magnitude_spectrum.shape) / 2
        r = np.sqrt((x - center[1])**2 + (y - center[0])**2)
        r = r.astype(int)

        # Ensure r doesn't exceed num_bins
        r_flat = r.ravel()
        mag_flat = magnitude_spectrum.ravel()
        
        # Calculate distance-based average
        tbin = np.bincount(r_flat, mag_flat)
        nr = np.bincount(r_flat)
        radial_profile = tbin / (nr + 1e-10) # Avoid division by zero
        
        return radial_profile[:self.num_bins]

    def extract(self, img_pil):
        # Preprocess
        img = np.array(img_pil.convert('L')) # Grayscale
        img = cv2.resize(img, (self.target_size, self.target_size))
        
        # FFT
        f = np.fft.fft2(img)
        fshift = np.fft.fftshift(f)
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-10)
        
        # 1D Radial Profile
        profile = self._azimuthal_integration(magnitude_spectrum)
        
        # Normalize profile to [0, 1] range for better convergence in SVM
        profile = (profile - np.min(profile)) / (np.max(profile) - np.min(profile) + 1e-10)
        
        return profile

class SemanticFeatureExtractor:
    """Extracts 512-d CLIP embeddings."""
    def __init__(self, model_name="openai/clip-vit-base-patch32", device="cpu"):
        self.device = device
        self.model = CLIPModel.from_pretrained(model_name).to(device)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model.eval()

    def extract(self, img_pil):
        inputs = self.processor(images=img_pil, return_tensors="pt").to(self.device)
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
        
        # Normalize features
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return image_features.cpu().numpy().flatten()

print("Feature Extractors initialized.")

In [ ]:
def create_dummy_fusion_data(n_samples=100):
    """
    Creates dummy AI and Real images and extracts features.
    Real images: Smooth, organic patterns.
    AI images: High-frequency 'staircase' noise artifacts in FFT.
    """
    spectral_ext = SpectralFeatureExtractor()
    semantic_ext = SemanticFeatureExtractor(device=device)
    
    features_list = []
    labels = []
    
    print(f"Generating and extracting features for {n_samples} samples...")
    for i in tqdm(range(n_samples)):
        is_ai = i < (n_samples // 2)
        
        # Create base image
        img_arr = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
        if not is_ai:
            # Real: Blur to simulate natural smoothness
            img_arr = cv2.GaussianBlur(img_arr, (15, 15), 0)
        else:
            # AI: Add periodic artifacts (checkboard/grid)
            for x in range(0, 256, 8):
                img_arr[:, x, :] = np.clip(img_arr[:, x, :] + 50, 0, 255)
            for y in range(0, 256, 8):
                img_arr[y, :, :] = np.clip(img_arr[y, :, :] + 50, 0, 255)

        img_pil = Image.fromarray(img_arr)
        
        # Extract features
        spec_feat = spectral_ext.extract(img_pil)  # 128-d
        sem_feat = semantic_ext.extract(img_pil)   # 512-d
        
        # CONCATENATE
        fused_feat = np.concatenate([spec_feat, sem_feat]) # 640-d
        
        features_list.append(fused_feat)
        labels.append(1 if is_ai else 0)
        
    return np.array(features_list), np.array(labels)

# Generate dataset
X, y = create_dummy_fusion_data(n_samples=100)
print(f"Feature matrix shape: {X.shape} (Combined Spectral 128 + Semantic 512)")

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. SVM Classification (RBF Kernel)
print("\n--- SVM CLASSIFIER ---")
svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm, target_names=['Real', 'AI']))

# 2. XGBoost Classification
print("\n--- XGBOOST CLASSIFIER ---")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb, target_names=['Real', 'AI']))

# Feature Importance (Top 10)
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.title("Feature Importance (First 128: Spectral, 128-640: Semantic)")
plt.bar(range(20), importances[indices[:20]], align='center')
plt.xticks(range(20), indices[:20], rotation=90)
plt.tight_layout()
plt.show()

## Summary of the Fusion Architecture

As specified in the research papers, this pipeline implements a **Multi-Modal Detection Framework**:

1.  **Spectral Analysis (Low-Level)**:
    - AI images generated by models like Stable Diffusion often exhibit "grid artifacts" or "staircase effects" in the Fourier domain.
    - By converting the 2D Spectrum into a **1D Radial Profile** using Azimuthal Integration, we condense thousands of spectral peaks into a 128-dimensional vector that highlights unnatural energy spikes.

2.  **Semantic Analysis (High-Level)**:
    - While spectral features catch low-level math errors, AI often fails at semantic coherence (e.g., lighting inconsistencies, anatomical errors).
    - **CLIP (ViT-B/32)** provides a 512-dimensional embedding that captures the "vision-language" alignment, helping the model identify images that don't match known "natural" semantic patterns.

3.  **Feature Concatenation & Classification**:
    - Combining these (640-d vector) allows the classifiers (**SVM** and **XGBoost**) to weigh both physical artifacts and semantic logic. 
    - **XGBoost** further provides "Feature Importance" to see which spectral frequencies or semantic dimensions contributed most to the detection.

In [ ]:
# Contoh pengambilan metrik kontinu (Probability Scores)
# Ini adalah inti dari No-Reference IQA: memberikan skor numerik untuk 'keaslian'

y_probs_svm = svm_model.predict_proba(X_test)[:, 1] # Kolom 1 adalah probabilitas 'AI'
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("--- Contoh Skor Kontinu (AI-Probability) ---")
for i in range(5):
    true_label = "AI" if y_test[i] == 1 else "Real"
    print(f"Sample {i} [{true_label}] -> SVM Score: {y_probs_svm[i]:.4f}, XGB Score: {y_probs_xgb[i]:.4f}")

# Visualisasi Distribusi Skor Kontinu
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(y_probs_svm[y_test==0], bins=20, alpha=0.5, label='Real', color='blue')
plt.hist(y_probs_svm[y_test==1], bins=20, alpha=0.5, label='AI', color='red')
plt.title("SVM Probability Distribution")
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(y_probs_xgb[y_test==0], bins=20, alpha=0.5, label='Real', color='blue')
plt.hist(y_probs_xgb[y_test==1], bins=20, alpha=0.5, label='AI', color='red')
plt.title("XGBoost Probability Distribution")
plt.legend()

plt.tight_layout()
plt.show()